# AST based evaluation for Model responses Single-turn Single tool call

In [13]:
import json
import pandas as pd
import re
from typing import List, Dict, Any

#### Helper Functions for AST ####
def standardize_string(input_string: str) -> str:
    # Standardizes strings by removing spaces/punctuation and converting to lowercase
    regex_string = r"[ \,\.\/\-\_\*\^]"
    return re.sub(regex_string, "", input_string).lower().replace("'", '"')

def is_valid_json(json_str: str) -> tuple[bool, str]:
    # Checks if a string is valid JSON and parsablegr
    try:
        json.loads(json_str)
        return True, ""
    except json.JSONDecodeError as e:
        return False, f"structural_completeness_error:invalid_json_syntax" # return error message

def normalize_none_string(text: str) -> str:
    """Normalize 'None (No function calls)' strings by removing quotes."""
    if isinstance(text, str):
        text = text.strip()
        # Remove surrounding quotes if present
        if text.startswith('"') and text.endswith('"'):
            text = text[1:-1]
        if text.startswith("'") and text.endswith("'"):
            text = text[1:-1]
    return text

def check_empty_ground_truth(ground_truth: Any) -> dict:
    # Handle nan values from pandas
    if pd.isna(ground_truth):
        return {
            "valid": False,
            "error": [],
            "special_case": "no_function_calls"
        }

    # Checks for empty, normal text, or invalid ground truth cases
    if ground_truth is None or ground_truth == "None":
        return {
            "valid": True,
            "error": ["Ground truth is None"],
            "error_type": "empty_ground_truth:none"
        }
    
    if isinstance(ground_truth, str):
        stripped = normalize_none_string(ground_truth)
        # Valid "no function calls" cases
        if stripped in ["", "[]", "[{}]", "None (No function calls)", "None"]:
            return {
                "valid": True,
                "error": [],
                "special_case": "no_function_calls_required"
            }
        # checking ground truth - Is it a valid JSON string (function call)?
        is_valid, error_msg = is_valid_json(stripped)
        if is_valid:
            return {"valid": True, "error": [], "special_case": "function_call_json"}
        elif stripped.startswith('[') or stripped.startswith('{'):
            # Looks like JSON but is invalid
            return {
                "valid": False, 
                "error":[error_msg],
                "error_type": "structural_completeness_error:invalid_json_format"
            }
        
        # FIXED: If it's not JSON and doesn't look like JSON, treat as normal text response
        # This was correct before
        return {
            "valid": True,
            "error": [],
            "special_case": "normal_text_response"
        }
    
    # If it's not a string, check if it's a dict/list (JSON loaded already)
    if isinstance(ground_truth, (dict, list)):
        return {"valid": True, "error": [], "special_case": "function_call_json"}
    
    # Anything else is invalid
    return {
        "valid": False,
        "error": [f"Ground truth is invalid or unsupported type: {type(ground_truth)}"],
        "error_type": "empty_ground_truth:invalid_type"
    }

def validate_parameters(model_args, expected_args_schema, param_path="arguments"):
    """
    model_args:      The arguments dict/list from model output, at any level.
    expected_args_schema:  The expected argument schema (dict of param: expected value or structure).
    param_path:      For error messages, the path in the object tree.
    """
    #print(model_args, expected_args_schema) 
    
    # Handle if expected_args_schema is a list (for array cases)
    if isinstance(expected_args_schema, list):
        if not isinstance(model_args, list):
            return {"valid": False, "error": [f"{param_path} should be a list. Got: {type(model_args).__name__}"], "error_type": "Parameter Filling Errors:type_mismatch"}
        if len(model_args) != len(expected_args_schema):
            return {"valid": False, "error": [f"{param_path} length mismatch. Expected {len(expected_args_schema)}, got {len(model_args)}"], "error_type": "Parameter Filling Errors:array_length_mismatch"}
        # Validate each item recursively
        for i, (model_item, expected_item) in enumerate(zip(model_args, expected_args_schema)):
            res = validate_parameters(model_item, expected_item, f"{param_path}[{i}]")
            if not res["valid"]:
                return res
        return {"valid": True, "error": []}

    # Must be dict at this point
    if not isinstance(model_args, dict) or not isinstance(expected_args_schema, dict):
        if model_args != expected_args_schema:
            #print(123)
            return {"valid": False, "error": [f"{param_path} value mismatch. Expected: {repr(expected_args_schema)}, got: {repr(model_args)}"], "error_type": "Parameter Filling Errors:value_mismatch"}
        return {"valid": True, "error": []}
    
    # Collect all errors (both hard and soft) before deciding
    hard_errors = []
    soft_errors = []

    # Required/optional check for dict keys
    for key, expected_val in expected_args_schema.items():
        # Handle optional parameter (if "" is in list of possible values)
        is_optional = False
        possible_vals = expected_val if isinstance(expected_val, list) else [expected_val]
        if "" in possible_vals or None in possible_vals or "null" in possible_vals:
            is_optional = True

        if key not in model_args:
            if not is_optional:
                return {"valid": False, "error": [f"Missing required parameter: {param_path}.{key}"], "error_type": "Parameter Filling Errors:missing_required"}
            else:
                continue  # OK if optional

        # Check value type and recurse if needed
        model_val = model_args[key]

        # **NEW: Add explicit type checking for arrays**
        if isinstance(expected_val, list) and not isinstance(model_val, list):
            # Check if expected_val represents an array schema (all items are the same type/structure)
            # This handles cases where expected_val is like ["string_type"] indicating array of strings
            if len(set(str(type(v)) for v in expected_val if v != "")) == 1:
                return {"valid": False, "error": [f"{param_path}.{key} should be an array. Expected: {expected_val}, got: {type(model_val).__name__} '{model_val}'"], "error_type": "Parameter Filling Errors:type_mismatch_array_expected"}

        # **NEW: Add explicit type checking when model provides array but schema expects scalar**
        if isinstance(model_val, list) and not isinstance(expected_val, list):
            return {"valid": False, "error": [f"{param_path}.{key} should be a scalar value, not an array. Expected: {expected_val}, got: {model_val}"], "error_type": "Parameter Filling Errors:type_mismatch_scalar_expected"}

        # ENUM/anyOf
        if isinstance(expected_val, list) and not isinstance(model_val, (dict, list)):
            # Acceptable values in list
            std_model_val = model_val
            std_possible = [v for v in expected_val if v != ""]
            if std_model_val not in std_possible:
                return {"valid": False, "error": [f"{param_path}.{key} value mismatch. Expected one of: {std_possible}, got: {model_val}"], "error_type": "Parameter Filling Errors:enum_value_mismatch"}
            continue

        # Nested dict
        if isinstance(model_val, dict) and isinstance(expected_val, dict):
            res = validate_parameters(model_val, expected_val, f"{param_path}.{key}")
            if not res["valid"]:
                return res
            continue

        # List (array)
        if isinstance(model_val, list) and isinstance(expected_val, list):
            res = validate_parameters(model_val, expected_val, f"{param_path}.{key}")
            if not res["valid"]:
                return res
            continue

        # Scalar value (string, number, bool, etc.)
        if isinstance(model_val, str) and isinstance(expected_val, str):
            # SOFT MATCH for free-form text (not strict)
            if len(expected_val) > 10 or len(model_val) > 10:  # arbitrary threshold for 'long' text
                # Accept if the model's value is a substring, or vice versa (soft)
                if not ((model_val in expected_val) or (expected_val in model_val)):
                    soft_errors.append(f"{param_path}.{key} textual difference. Expected: {expected_val}, got: {model_val}")
            else:
                # For short strings, use strict match - this is a HARD error
                if model_val != expected_val and expected_val != "":
                    return {"valid": False, "error": [f"{param_path}.{key} value mismatch. Expected: {expected_val}, got: {model_val}"], "error_type": "Parameter Filling Errors:value_mismatch"}
        else:
            if model_val != expected_val and expected_val != "":
                return {"valid": False, "error": [f"{param_path}.{key} value mismatch. Expected: {expected_val}, got: {model_val}"], "error_type": "Parameter Filling Errors:value_mismatch"}


    # Check for unexpected parameters (not in schema)
    for key in model_args:
        if key not in expected_args_schema:
            return {"valid": False, "error": [f"Unexpected parameter: {param_path}.{key}"], "error_type": "Parameter Filling Errors:unexpected_param"}

    # If we have hard errors, return them first
    if hard_errors:
        return {"valid": False, "error": hard_errors, "error_type": "Parameter Filling Errors:value_mismatch"}

    # If we have soft errors, return them but mark as valid
    if soft_errors:
        return {"valid": True, "error": soft_errors, "error_type": "Parameter Filling Errors:soft_text_difference"}

    return {"valid": True, "error": []}

def string_checker(param: str, model_output: str, possible_answer: List[Any]) -> dict:
    # Validates string values (case-insensitive)
    standardized_model_output = standardize_string(model_output)
    standardized_possible_answer = [
        standardize_string(val) if isinstance(val, str) else val
        for val in possible_answer
    ]
    if standardized_model_output not in standardized_possible_answer:
        return {
            "valid": False,
            "error": [f"Invalid value for parameter {repr(param)}: {repr(model_output)}. Expected one of {possible_answer}."],
            "error_type": "value_error:string"
        }
    return {"valid": True, "error": []}

def list_checker(param: str, model_output: List[Any], possible_answer: List[List[Any]]) -> dict:
    # Recursively validates lists, handling nested lists and dictionaries
    standardized_model_output = []
    for item in model_output:
        if isinstance(item, str):
            standardized_model_output.append(standardize_string(item))
        elif isinstance(item, dict):
            # Check nested dictionary
            for possible_answer_item in possible_answer:
                result = dict_checker(param, item, possible_answer_item)
                if result["valid"]:
                    standardized_model_output.append(item)
                    break
            else:
                return {
                    "valid": False,
                    "error": [f"Invalid nested dictionary in list for parameter {repr(param)}: {repr(item)}."],
                    "error_type": "value_error:nested_dict"
                }
        elif isinstance(item, list):
            # Check nested list
            for possible_answer_item in possible_answer:
                result = list_checker(param, item, possible_answer_item)
                if result["valid"]:
                    standardized_model_output.append(item)
                    break
            else:
                return {
                    "valid": False,
                    "error": [f"Invalid nested list for parameter {repr(param)}: {repr(item)}."],
                    "error_type": "value_error:nested_list"
                }
        else:
            standardized_model_output.append(item)

    standardized_possible_answer = []
    for answer in possible_answer:
        standardized_answer = []
        for item in answer:
            if isinstance(item, str):
                standardized_answer.append(standardize_string(item))
            else:
                standardized_answer.append(item)
        standardized_possible_answer.append(standardized_answer)

    if standardized_model_output not in standardized_possible_answer:
        return {
            "valid": False,
            "error": [f"Invalid value for parameter {repr(param)}: {repr(model_output)}. Expected one of {possible_answer}."],
            "error_type": "value_error:list"
        }
    return {"valid": True, "error": []}

def dict_checker(param: str, model_output: Dict[str, Any], possible_answer: List[Dict[str, Any]]) -> dict:
    # Recursively validates dictionaries, handling nested dictionaries and lists
    result = {"valid": False, "error": [], "error_type": "dict_checker:unclear"}
    for answer in possible_answer:
        if answer == "":
            continue
        result = {"valid": True, "error": [], "error_type": "dict_checker:unclear"}
        for key, value in model_output.items():
            if key not in answer:
                result["valid"] = False
                result["error"].append(f"Unexpected dict key parameter: {repr(key)}.")
                result["error_type"] = "value_error:dict_key"
                break
            possible_values = answer[key] if isinstance(answer[key], list) else [answer[key]]
            if isinstance(value, dict):
                # Recursive dictionary check
                sub_result = dict_checker(f"{param}.{key}", value, possible_values)
                if not sub_result["valid"]:
                    result["valid"] = False
                    result["error"].extend(sub_result["error"])
                    result["error_type"] = sub_result["error_type"]
                    break
            elif isinstance(value, list):
                # Recursive list check
                sub_result = list_checker(f"{param}.{key}", value, possible_values)
                if not sub_result["valid"]:
                    result["valid"] = False
                    result["error"].extend(sub_result["error"])
                    result["error_type"] = sub_result["error_type"]
                    break
            else:
                # Handle strings and other types
                standardized_value = standardize_string(value) if isinstance(value, str) else value
                standardized_possible_values = [
                    standardize_string(val) if isinstance(val, str) else val
                    for val in possible_values
                ]
                if standardized_value not in standardized_possible_values:
                    result["valid"] = False
                    result["error"].append(
                        f"Invalid value for parameter {repr(key)}: {repr(value)}. Expected one of {possible_values}."
                    )
                    result["error_type"] = "value_error:dict_value"
                    break
        if result["valid"]:
            # Check for missing required keys
            for key in answer:
                if key not in model_output and "" not in (answer[key] if isinstance(answer[key], list) else [answer[key]]):
                    result["valid"] = False
                    result["error"].append(f"Missing dict key parameter: {repr(key)}.")
                    result["error_type"] = "value_error:dict_key"
                    break
        if result["valid"]:
            return result
    return result

def simple_function_checker(model_output: Dict[str, Any], ground_truth: Dict[str, Any]) -> dict:
    result = {"valid": True, "error": [], "error_type": "simple_function_checker:unclear"}
    
    # 1. Parse model_output if it's a string
    if isinstance(model_output, str):
        try:
            model_output = json.loads(model_output) # Convert string to JSON
        except json.JSONDecodeError as e:
            return {
                "valid": False,
                "error": [f"structural_completeness_error - {str(e)}"],
                "error_type": "simple_function_checker_structural_completeness_error:invalid_json"
            }
    
    # 2. Extract function name and parameters from possible_answer
    if "name" in ground_truth:
        expected_func_name = ground_truth["name"]
        expected_params = ground_truth["arguments"]
    else:
        # Legacy format: {"func_name": {"arguments": {...}}}
        expected_func_name = list(ground_truth.keys())[0]
        expected_params = ground_truth[expected_func_name]["arguments"]
    
    # 3. Tool Selection Accuracy
    if "name" in model_output and "arguments" in model_output:
        if model_output["name"] != expected_func_name:
            result["valid"] = False
            result["error"].append(f"Tool Selection Accuracy: Function name mismatch - expected {repr(expected_func_name)}, got {repr(model_output['name'])}.")
            result["error_type"] = "tool_selection_accuracy:simple_function_checker:wrong_func_name"
            return result
        model_params = model_output["arguments"] 
    elif expected_func_name in model_output: 
        model_params = model_output[expected_func_name]["arguments"] 
    else:
        result["valid"] = False
        result["error"].append(f"Tool Selection Accuracy: Function name {repr(expected_func_name)} not found in model output.")
        result["error_type"] = "tool_selection_accuracy:simple_function_checker:function_not_found"
        return result

    # Use expected_params (not possible_answer[expected_func_name]["arguments"])
    possible_params = expected_params

    # 4. Validate parameters
    result = validate_parameters(model_params, possible_params)
    #print(result)
    return result  # Add this missing return statement

# Main AST checker function that integrates all the logic
def ast_checker(model_output: Any, ground_truth: Any) -> dict:
    # Check for empty/invalid/normal text/JSON ground truth
    empty_check = check_empty_ground_truth(ground_truth)
    if not empty_check["valid"]: 
        return empty_check
    
    # 1. No function calls expected
    if "special_case" in empty_check and empty_check["special_case"] == "no_function_calls_required":
        # Check if model output is empty/nan/none
        normalized_output = normalize_none_string(model_output) if isinstance(model_output, str) else model_output
        if pd.isna(model_output) or model_output is None or (
            isinstance(model_output, str) and 
            normalized_output in ["", "[]", "[{}]", "None", "None (No function calls)"]):
            return {
                "valid": True,
                "error": [],
                "error_type": "no_function_calls_match"
            }
        else:
            return {
                "valid": False,
                "error": ["Ground truth expects no function calls, but model provided function calls"],
                "error_type": "function_calls_when_none_expected"
            }
        
    # 2. Normal text response expected
    if "special_case" in empty_check and empty_check["special_case"] == "normal_text_response":
        # FIXED: Check if model output is a function call when text is expected
        if isinstance(model_output, str):
            # Check if model output looks like a function call
            normalized_output = normalize_none_string(model_output)
            is_model_json_valid, _ = is_valid_json(normalized_output)
            if is_model_json_valid:
                # Model provided function call but GT expects text
                return {
                    "valid": False,
                    "error": ["Ground truth expects normal text response, but model provided function calls"],
                    "error_type": "function_calls_when_text_expected"
                }
            
            # Both are text, compare
            if model_output.strip() == str(ground_truth).strip():
                return {"valid": True, "error": [], "error_type": "normal_text_exact_match"}
            else:
                return {
                    "valid": True,  # Keep as warning, not error
                    "error": [f"Text difference (warning): Expected '{ground_truth}', got '{model_output}'"],
                    "error_type": "normal_text_difference_warning"
                }
        else:
            return {
                "valid": False,
                "error": [f"Expected normal text response, but model output is not string: {model_output}"],
                "error_type": "model_output_not_text"  
            }
        
    # 3. Check if model output indicates no function calls but ground truth expects them
    if isinstance(model_output, str):
        normalized_output = normalize_none_string(model_output)
        if normalized_output in ["", "[]", "[{}]", "None", "None (No function calls)"]:
            # Model says no function calls, but we expect function calls
            return {
                "valid": False,
                "error": ["Model provided no function calls, but ground truth expects function calls"],
                "error_type": "no_function_calls_when_expected"
            }
        
        # FIXED: Add check for when model provides text but GT expects function calls
        is_model_valid_json, _ = is_valid_json(normalized_output)
        if not is_model_valid_json and not (normalized_output.startswith('[') or normalized_output.startswith('{')):
            # Model provided text response but GT expects function calls
            return {
                "valid": False,
                "error": ["Model provided text response, but ground truth expects function calls"],
                "error_type": "text_when_function_calls_expected"
            }
            
    # 4. Parse JSON strings
    try:
        if isinstance(model_output, str):
            is_valid, error = is_valid_json(model_output)
            if not is_valid:
                return {
                    "valid": False,
                    "error": [error],
                    "error_type": "structural_completeness_error:invalid_json"
                }
            model_output = json.loads(model_output)
            
        if isinstance(ground_truth, str):
            ground_truth = json.loads(ground_truth)

    except Exception as e:
        return {
            "valid": False,
            "error": [f"Structural Completeness Error: JSON parsing failed - {str(e)}"],
            "error_type": "structural_completeness_error:json_parse_error"
        }

    # 5. Validate structure
    if not isinstance(model_output, list):
        model_output = [model_output]  # Convert single function call to list
    if not isinstance(ground_truth, list):
        ground_truth = [ground_truth]  # Convert single function call to list
        
    # Check each function call has required fields
    if not all(isinstance(item, dict) and ("name" in item or len(item) == 1) for item in model_output):
        return {
            "valid": False,
            "error": ["Model output is not a valid list of function calls"],
            "error_type": "structural_completeness_error:invalid_format"
        }
    if not all(isinstance(item, dict) and ("name" in item or len(item) == 1) for item in ground_truth):
        return {
            "valid": False,
            "error": ["Ground truth is not a valid list of function calls"],
            "error_type": "structural_completeness_error:invalid_format"
        }

    # 6. Check counts and determine call type
    if len(model_output) != len(ground_truth):
        return {
            "valid": False,
            "error": [f"Wrong number of functions: got {len(model_output)}, expected {len(ground_truth)}."],
            "error_type": "function_count_mismatch"
        }
    
    # Only handle single function calls
    if len(ground_truth) != 1:
        return {
            "valid": False,
            "error": [f"Parallel function calls not supported. Expected 1, got {len(ground_truth)}."],
            "error_type": "Parallel_function_calls_not_supported"
        }
        
    return simple_function_checker(model_output[0], ground_truth[0])
    
def evaluate_excel_data(file_path: str) -> List[Dict[str, Any]]:
    # Reads Excel file and evaluates each row
    df = pd.read_excel(file_path)
    results = []
    for index, row in df.iterrows():
        model_output = row["Actual_model_call"]
        ground_truth = row["Ground_truth"]
        result = ast_checker(model_output, ground_truth)
        results.append({
            "row_index": index,
            "model_output": model_output,
            "ground_truth": ground_truth,
            "evaluation": result
        })
    return results

# Example usage
if __name__ == "__main__":
    results = evaluate_excel_data("D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2.Mistral_single_FC\\mistral_latest_results\\results_mistralai\\hp_fine_tuned_ministral-8B-Instruct-2410_pipeline.xlsx")
    for result in results:
        print(f"Row {result['row_index']}:")
        print(f"Model Output: {result['model_output']}")
        print(f"Ground Truth: {result['ground_truth']}")
        print(f"Evaluation: {result['evaluation']}")
        # print(f"Accuracy: {result['accuracy']}")
        # print(f"Precision: {result['precision']}")
        print()

# --- Stats calculation for Single-Turn Evaluation ---
single_count = 0
single_errors = 0
single_error_types = {}
total_valid = 0
total_invalid = 0

for result in results:
    eval_result = result["evaluation"]
    single_count += 1
    
    if eval_result["valid"]:
        total_valid += 1
    else:
        single_errors += 1
        total_invalid += 1
        etype = eval_result.get("error_type", "unknown")
        single_error_types[etype] = single_error_types.get(etype, 0) + 1

# Calculate success rate
success_rate = (total_valid / single_count * 100) if single_count > 0 else 0
error_rate = (single_errors / single_count * 100) if single_count > 0 else 0

print("\n---- Single-Turn Evaluation Statistics ----")
print(f"Total evaluations: {single_count}")
print(f"Successful evaluations: {total_valid} ({success_rate:.2f}%)")
print(f"Failed evaluations: {single_errors} ({error_rate:.2f}%)")
print("\nError breakdown by type:")
for error_type, count in sorted(single_error_types.items()):
    percentage = (count / single_errors * 100) if single_errors > 0 else 0
    print(f"  {error_type}: {count} ({percentage:.1f}%)")

# Additional breakdown by error categories
structural_errors = 0
tool_selection_errors = 0
parameter_errors = 0
other_errors = 0

for error_type, count in single_error_types.items():
    if "structural_completeness_error" in error_type:
        structural_errors += count
    elif "tool_selection_accuracy" in error_type:
        tool_selection_errors += count
    elif "parameter" in error_type.lower() or "filling" in error_type:
        parameter_errors += count
    else:
        other_errors += count

print(f"\nError Category Summary:")
print(f"  Structural/JSON errors: {structural_errors}")
print(f"  Tool selection errors: {tool_selection_errors}")
print(f"  Parameter filling errors: {parameter_errors}")
print(f"  Other errors: {other_errors}")


#########################

def classify_ml_metrics(evaluation_result, ground_truth):
    """
    Classify AST checker results into ML metrics categories:
    TP, TN, FP, FN based on the given definitions
    """
    error_type = evaluation_result.get("error_type", "")
    is_valid = evaluation_result.get("valid", False)
    
    # Check if ground truth is null function call or normal text
    gt_check = check_empty_ground_truth(ground_truth)
    is_gt_null_or_text = "special_case" in gt_check and gt_check["special_case"] in [
        "no_function_calls", "no_function_calls_required", "normal_text_response"
    ]
    is_gt_function_call = "special_case" in gt_check and gt_check["special_case"] == "function_call_json"
    
    # TRUE NEGATIVE: label is null/text, prediction is also null/text (both non-function calls)
    if is_gt_null_or_text and error_type in [
        "no_function_calls_match", 
        "normal_text_exact_match", 
        "normal_text_difference_warning"
    ]:
        return "TN"
    
    # FALSE POSITIVE Case 1: label is null/text, model predicted function call
    if is_gt_null_or_text and error_type == "function_calls_when_none_expected":
        return "FP"
    
    # FALSE NEGATIVE: label is function call, model predicted null/text
    if is_gt_function_call and error_type in [
        "no_function_calls_when_expected",
        "text_when_function_calls_expected"
    ]:
        return "FN"
    
    # TRUE POSITIVE: label is function call, prediction is valid function call and matches
    if is_gt_function_call and is_valid and error_type not in [
        "function_calls_when_none_expected",
        "no_function_calls_when_expected",
        "function_calls_when_text_expected", 
        "text_when_function_calls_expected",
        "model_output_not_text"
    ]:
        return "TP"
    
    # FALSE POSITIVE Case 2: label is function call, but prediction has mismatch
    if is_gt_function_call and (not is_valid or error_type in [
        "tool_selection_accuracy:simple_function_checker:wrong_func_name",
        "tool_selection_accuracy:simple_function_checker:function_not_found",
        "Parameter Filling Errors:missing_required",
        "Parameter Filling Errors:unexpected_param", 
        "Parameter Filling Errors:type_mismatch",
        "Parameter Filling Errors:value_mismatch",
        "Parameter Filling Errors:enum_value_mismatch",
        "Parameter Filling Errors:type_mismatch_array_expected",
        "Parameter Filling Errors:type_mismatch_scalar_expected",
        "structural_completeness_error:invalid_json",
        "structural_completeness_error:invalid_json_format",
        "structural_completeness_error:json_parse_error",
        "function_calls_when_text_expected"
    ]):
        return "FP"
    
    # Default fallback - treat as FP if uncertain
    return "FP"

def calculate_ml_metrics(results):
    """
    Calculate accuracy, precision, recall, and F1 score from classification results
    """
    # Classify each result
    classifications = []
    for result in results:
        classification = classify_ml_metrics(result["evaluation"], result["ground_truth"])
        classifications.append(classification)
    
    # Count each category
    tp_count = classifications.count("TP")
    tn_count = classifications.count("TN")
    fp_count = classifications.count("FP")
    fn_count = classifications.count("FN")
    
    total = len(classifications)
    
    # Calculate metrics
    accuracy = (tp_count + tn_count) / total if total > 0 else 0
    precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0
    recall = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "total_samples": total,
        "true_positives": tp_count,
        "true_negatives": tn_count,
        "false_positives": fp_count,
        "false_negatives": fn_count,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "classifications": classifications
    }

# Add this code after the existing statistics calculation
print("\n---- ML Metrics Classification ----")

# Calculate ML metrics
ml_metrics = calculate_ml_metrics(results)

print(f"Total samples: {ml_metrics['total_samples']}")
print(f"True Positives (TP): {ml_metrics['true_positives']}")
print(f"True Negatives (TN): {ml_metrics['true_negatives']}")
print(f"False Positives (FP): {ml_metrics['false_positives']}")
print(f"False Negatives (FN): {ml_metrics['false_negatives']}")
print()
print(f"Accuracy: {ml_metrics['accuracy']:.4f} ({ml_metrics['accuracy']*100:.2f}%)")
print(f"Precision: {ml_metrics['precision']:.4f} ({ml_metrics['precision']*100:.2f}%)")
print(f"Recall: {ml_metrics['recall']:.4f} ({ml_metrics['recall']*100:.2f}%)")
print(f"F1 Score: {ml_metrics['f1_score']:.4f} ({ml_metrics['f1_score']*100:.2f}%)")


def calculate_custom_metrics(results):
    """
    Calculate the three custom metrics for tool usage evaluation
    """
    # Initialize counters
    total_cases = 0
    correct_tool_calls = 0
    correct_no_tool_responses = 0
    
    total_tool_predictions = 0
    correct_tool_name_predictions = 0
    
    total_correct_tool_names = 0  
    correct_parameter_predictions = 0
    
    # Process each result
    for result in results:
        evaluation = result["evaluation"]
        ground_truth = result["ground_truth"]
        error_type = evaluation.get("error_type", "")
        is_valid = evaluation.get("valid", False)
        
        # Check ground truth type
        gt_check = check_empty_ground_truth(ground_truth)
        is_gt_function_call = "special_case" in gt_check and gt_check["special_case"] == "function_call_json"
        is_gt_no_tool = "special_case" in gt_check and gt_check["special_case"] in [
            "no_function_calls", "no_function_calls_required", "normal_text_response"
        ]
        
        total_cases += 1
        
        # 1. Tool Usage Detection Rate
        if is_gt_function_call:
            # Model should predict tool call
            if error_type not in ["no_function_calls_when_expected", "text_when_function_calls_expected"]:
                correct_tool_calls += 1
        elif is_gt_no_tool:
            # Model should predict no tool call  
            if error_type in ["no_function_calls_match", "normal_text_exact_match", "normal_text_difference_warning"]:
                correct_no_tool_responses += 1
        
        # 2. Tool Selection Robustness (only count when model predicted tool call)
        model_predicted_tool = error_type not in [
            "no_function_calls_when_expected", 
            "text_when_function_calls_expected",
            "no_function_calls_match",
            "normal_text_exact_match", 
            "normal_text_difference_warning"
        ]
        
        if model_predicted_tool:
            total_tool_predictions += 1
            # Check if tool name is correct (no tool selection errors)
            if error_type not in [
                "tool_selection_accuracy:simple_function_checker:wrong_func_name",
                "tool_selection_accuracy:simple_function_checker:function_not_found"
            ]:
                correct_tool_name_predictions += 1
                total_correct_tool_names += 1
                
                # 3. Parameter Completeness Rate (only for correct tool names)
                if is_valid and error_type not in [
                    "Parameter Filling Errors:missing_required",
                    "Parameter Filling Errors:unexpected_param",
                    "Parameter Filling Errors:type_mismatch", 
                    "Parameter Filling Errors:value_mismatch",
                    "Parameter Filling Errors:enum_value_mismatch",
                    "Parameter Filling Errors:type_mismatch_array_expected",
                    "Parameter Filling Errors:type_mismatch_scalar_expected",
                    "structural_completeness_error:invalid_json",
                    "structural_completeness_error:invalid_json_format",
                    "structural_completeness_error:json_parse_error"
                ]:
                    correct_parameter_predictions += 1
    
    # Calculate metrics
    tool_usage_detection_rate = (correct_tool_calls + correct_no_tool_responses) / total_cases if total_cases > 0 else 0
    tool_selection_robustness = correct_tool_name_predictions / total_tool_predictions if total_tool_predictions > 0 else 0
    parameter_completeness_rate = correct_parameter_predictions / total_correct_tool_names if total_correct_tool_names > 0 else 0
    
    return {
        "total_cases": total_cases,
        "correct_tool_calls": correct_tool_calls,
        "correct_no_tool_responses": correct_no_tool_responses,
        "total_tool_predictions": total_tool_predictions,
        "correct_tool_name_predictions": correct_tool_name_predictions,
        "total_correct_tool_names": total_correct_tool_names,
        "correct_parameter_predictions": correct_parameter_predictions,
        "tool_usage_detection_rate": tool_usage_detection_rate,
        "tool_selection_robustness": tool_selection_robustness,
        "parameter_completeness_rate": parameter_completeness_rate
    }

# Save results back to Excel with ML metrics columns
def save_results_with_all_metrics(file_path, results, ml_metrics, custom_metrics):
    """
    Save results with both ML metrics and custom metrics to Excel
    """
    # Read the file with ML metrics (if it exists) or original file
    ml_file = file_path.replace('.xlsx', '_with_ml_metrics.xlsx')
    try:
        df = pd.read_excel(ml_file)
    except:
        df = pd.read_excel(file_path)
        # Add ML metrics columns if not present
        df['TP'] = 0
        df['TN'] = 0  
        df['FP'] = 0
        df['FN'] = 0
        for i, classification in enumerate(ml_metrics['classifications']):
            if i < len(df):
                df.at[i, classification] = 1
    
    # Add custom metrics columns with binary indicators
    df['Tool_Usage_Detection_Rate'] = 0
    df['Tool_Selection_Robustness'] = 0  
    df['Parameter_Completeness_Rate'] = 0
    
    # Fill custom metrics columns
    for i, result in enumerate(results):
        if i >= len(df):
            break
            
        evaluation = result["evaluation"]
        ground_truth = result["ground_truth"]
        error_type = evaluation.get("error_type", "")
        is_valid = evaluation.get("valid", False)
        
        gt_check = check_empty_ground_truth(ground_truth)
        is_gt_function_call = "special_case" in gt_check and gt_check["special_case"] == "function_call_json"
        is_gt_no_tool = "special_case" in gt_check and gt_check["special_case"] in [
            "no_function_calls", "no_function_calls_required", "normal_text_response"
        ]
        
        # Tool Detection
        tool_detection_correct = False
        if is_gt_function_call and error_type not in ["no_function_calls_when_expected", "text_when_function_calls_expected"]:
            tool_detection_correct = True
        elif is_gt_no_tool and error_type in ["no_function_calls_match", "normal_text_exact_match", "normal_text_difference_warning"]:
            tool_detection_correct = True
        df.at[i, 'Tool_Usage_Detection_Rate'] = 1 if tool_detection_correct else 0

        # Tool Selection (only for cases where model predicted tool call)
        model_predicted_tool = error_type not in [
            "no_function_calls_when_expected", "text_when_function_calls_expected",
            "no_function_calls_match", "normal_text_exact_match", "normal_text_difference_warning"
        ]
        
        if model_predicted_tool and is_gt_function_call:
            tool_selection_correct = error_type not in [
                "tool_selection_accuracy:simple_function_checker:wrong_func_name",
                "tool_selection_accuracy:simple_function_checker:function_not_found"
            ]
            df.at[i, 'Tool_Selection_Robustness'] = 1 if tool_selection_correct else 0
            
            # Parameter Completeness (only for correct tool selections)
            if tool_selection_correct:
                parameter_correct = is_valid and error_type not in [
                    "Parameter Filling Errors:missing_required", "Parameter Filling Errors:unexpected_param",
                    "Parameter Filling Errors:type_mismatch", "Parameter Filling Errors:value_mismatch",
                    "Parameter Filling Errors:enum_value_mismatch", "Parameter Filling Errors:type_mismatch_array_expected",
                    "Parameter Filling Errors:type_mismatch_scalar_expected", "structural_completeness_error:invalid_json",
                    "structural_completeness_error:invalid_json_format", "structural_completeness_error:json_parse_error"
                ]
                df.at[i, 'Parameter_Completeness_Rate'] = 1 if parameter_correct else 0
    
    # Save to final Excel file
    output_file = file_path.replace('.xlsx', '_with_all_metrics.xlsx')
    df.to_excel(output_file, index=False)
    print(f"\nAll results saved to: {output_file}")
    return output_file

# Calculate custom metrics
custom_metrics = calculate_custom_metrics(results)

# Print custom metrics results
print("\n---- Custom Tool Usage Metrics ----")
print(f"1. Tool Usage Detection Rate: {custom_metrics['tool_usage_detection_rate']:.4f} ({custom_metrics['tool_usage_detection_rate']*100:.2f}%)")
print(f"   - Correct Tool Calls: {custom_metrics['correct_tool_calls']}")
print(f"   - Correct No-Tool Responses: {custom_metrics['correct_no_tool_responses']}")
print(f"   - Total Cases: {custom_metrics['total_cases']}")

print(f"\n2. Tool Selection Robustness: {custom_metrics['tool_selection_robustness']:.4f} ({custom_metrics['tool_selection_robustness']*100:.2f}%)")
print(f"   - Correct Tool Name Predictions: {custom_metrics['correct_tool_name_predictions']}")
print(f"   - Total Tool Call Predictions: {custom_metrics['total_tool_predictions']}")

print(f"\n3. Parameter Completeness Rate: {custom_metrics['parameter_completeness_rate']:.4f} ({custom_metrics['parameter_completeness_rate']*100:.2f}%)")
print(f"   - Correct Parameter Predictions: {custom_metrics['correct_parameter_predictions']}")
print(f"   - Total Correct Tool Names: {custom_metrics['total_correct_tool_names']}")

# Save results with all metrics
if __name__ == "__main__":
    final_output_file = save_results_with_all_metrics(
"D:\\Uni Study\\Sem-6\\6.My Code\\1.Single_turn\\2.Mistral_single_FC\\mistral_latest_results\\results_mistralai\\hp_fine_tuned_ministral-8B-Instruct-2410_pipeline.xlsx",
        results,
        ml_metrics, 
        custom_metrics
    )

# Optional: Show detailed breakdown
print("\n---- Detailed Classification Breakdown ----")
for i, (result, classification) in enumerate(zip(results, ml_metrics['classifications'])):
    error_type = result["evaluation"].get("error_type", "unknown")
    print(f"Row {i}: {classification} - {error_type}")

Row 0:
Model Output: { "name": "generate_sales_forecast", "arguments": { "forecast_period": { "start_date": "2024-01-07", "end_date": "2024-12-31", "forecast_type": "annual" }, "territories": [ "US", "EMEA" ], "product_categories": [ "electronics", "apparel" ] } }
Ground Truth: [{"name": "generate_sales_forecast", "arguments": {"forecast_period": {"start_date": "2024-07-01", "end_date": "2024-12-31"}, "territories": ["US", "EMEA"], "product_categories": ["electronics", "apparel"]}}]
Evaluation: {'valid': False, 'error': ['arguments.forecast_period.start_date value mismatch. Expected: 2024-07-01, got: 2024-01-07'], 'error_type': 'Parameter Filling Errors:value_mismatch'}

Row 1:
Model Output: { "name": "create_support_ticket", "arguments": { "customer_id": "CUST789", "issue_category": "technical", "priority": "high", "subject": "Intermittent Connection Drops During Peak Hours", "description": "Customer is experiencing intermittent connection drops during peak hours on the Premium Cloud 

# Statistical Evaluation

In [18]:
import numpy as np
from scipy import stats
import pandas as pd
from scipy.stats import bootstrap
import warnings
warnings.filterwarnings('ignore')

def calculate_cohens_d(group1, group2):
    """Calculate Cohen's d for effect size"""
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1 - 1) * np.var(group1, ddof=1) + (n2 - 1) * np.var(group2, ddof=1)) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std != 0 else 0

def bootstrap_confidence_interval(data, stat_func, confidence=0.95, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for a statistic"""
    rng = np.random.default_rng(42)  # Fixed seed for reproducibility
    
    def bootstrap_stat(data):
        return stat_func(data)
    
    # Perform bootstrap
    bootstrap_samples = []
    for _ in range(n_bootstrap):
        bootstrap_sample = rng.choice(data, size=len(data), replace=True)
        bootstrap_samples.append(bootstrap_stat(bootstrap_sample))
    
    # Calculate confidence interval
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_samples, 100 * alpha/2)
    upper = np.percentile(bootstrap_samples, 100 * (1 - alpha/2))
    
    return lower, upper, np.mean(bootstrap_samples)

def statistical_validation_analysis(results_list, model_names=None):
    """
    Perform statistical validation analysis on multiple models' results
    
    Args:
        results_list: List of results from different models/conditions
        model_names: List of names for each model (optional)
    """
    if model_names is None:
        model_names = [f"Model_{i+1}" for i in range(len(results_list))]
    
    print("=" * 60)
    print("STATISTICAL VALIDATION ANALYSIS")
    print("=" * 60)
    
    # Calculate metrics for each model
    all_metrics = []
    for i, results in enumerate(results_list):
        # ML Metrics
        ml_metrics = calculate_ml_metrics(results)
        
        # Custom Metrics  
        custom_metrics = calculate_custom_metrics(results)
        
        metrics_dict = {
            'model': model_names[i],
            'accuracy': ml_metrics['accuracy'],
            'precision': ml_metrics['precision'], 
            'recall': ml_metrics['recall'],
            'f1_score': ml_metrics['f1_score'],
            'tool_detection_rate': custom_metrics['tool_usage_detection_rate'],
            'tool_selection_robustness': custom_metrics['tool_selection_robustness'],
            'parameter_completeness_rate': custom_metrics['parameter_completeness_rate'],
            'sample_size': len(results)
        }
        all_metrics.append(metrics_dict)
    
    # Create DataFrame
    df_metrics = pd.DataFrame(all_metrics)
    print("\n1. PERFORMANCE SUMMARY")
    print("-" * 30)
    print(df_metrics.round(4))
    
    # If only one model, perform bootstrap analysis
    if len(results_list) == 1:
        print(f"\n2. BOOTSTRAP CONFIDENCE INTERVALS (95%) - {model_names[0]}")
        print("-" * 50)
        
        results = results_list[0]
        
        # Create binary arrays for bootstrap
        ml_metrics = calculate_ml_metrics(results)
        custom_metrics = calculate_custom_metrics(results)
        
        # Binary success arrays
        tp_array = [1 if classify_ml_metrics(r["evaluation"], r["ground_truth"]) == "TP" else 0 for r in results]
        accuracy_array = [1 if classify_ml_metrics(r["evaluation"], r["ground_truth"]) in ["TP", "TN"] else 0 for r in results]
        
        # Bootstrap confidence intervals
        metrics_to_bootstrap = [
            ('Accuracy', accuracy_array),
            ('Tool Detection Rate', []), # Will calculate from results
        ]
        
        # For accuracy
        acc_lower, acc_upper, acc_mean = bootstrap_confidence_interval(
            accuracy_array, lambda x: np.mean(x), confidence=0.95
        )
        
        print(f"Accuracy: {acc_mean:.4f} [{acc_lower:.4f}, {acc_upper:.4f}]")
        print(f"Tool Detection Rate: {custom_metrics['tool_usage_detection_rate']:.4f}")
        print(f"Tool Selection Robustness: {custom_metrics['tool_selection_robustness']:.4f}")
        print(f"Parameter Completeness: {custom_metrics['parameter_completeness_rate']:.4f}")
        
        return df_metrics
    
    # If multiple models, perform comparative analysis
    if len(results_list) >= 2:
        print(f"\n2. STATISTICAL SIGNIFICANCE TESTS (T-Tests)")
        print("-" * 45)
        
        # Compare first model with others
        baseline_results = results_list[0]
        baseline_name = model_names[0]
        
        # Create binary success arrays for baseline
        baseline_accuracy = [1 if classify_ml_metrics(r["evaluation"], r["ground_truth"]) in ["TP", "TN"] else 0 
                           for r in baseline_results]
        
        for i in range(1, len(results_list)):
            comparison_results = results_list[i]
            comparison_name = model_names[i]
            
            # Create binary success arrays for comparison model
            comparison_accuracy = [1 if classify_ml_metrics(r["evaluation"], r["ground_truth"]) in ["TP", "TN"] else 0 
                                 for r in comparison_results]
            
            # Perform t-test
            t_stat, p_value = stats.ttest_ind(baseline_accuracy, comparison_accuracy)
            
            # Calculate Cohen's d
            cohens_d = calculate_cohens_d(baseline_accuracy, comparison_accuracy)
            
            # Effect size interpretation
            if abs(cohens_d) < 0.2:
                effect_size = "negligible"
            elif abs(cohens_d) < 0.5:
                effect_size = "small"
            elif abs(cohens_d) < 0.8:
                effect_size = "medium"
            else:
                effect_size = "large"
            
            print(f"\n{baseline_name} vs {comparison_name}:")
            print(f"  T-statistic: {t_stat:.4f}")
            print(f"  P-value: {p_value:.4f}")
            print(f"  Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (α = 0.05)")
            print(f"  Cohen's d: {cohens_d:.4f} ({effect_size} effect)")
            
            # Performance difference
            baseline_acc = np.mean(baseline_accuracy)
            comparison_acc = np.mean(comparison_accuracy) 
            diff = comparison_acc - baseline_acc
            print(f"  Performance difference: {diff:+.4f} ({diff*100:+.2f}%)")
    
    print(f"\n3. BOOTSTRAP CONFIDENCE INTERVALS (95%)")
    print("-" * 40)
    
    for i, (results, name) in enumerate(zip(results_list, model_names)):
        # Create accuracy array
        accuracy_array = [1 if classify_ml_metrics(r["evaluation"], r["ground_truth"]) in ["TP", "TN"] else 0 
                         for r in results]
        
        # Bootstrap confidence interval
        lower, upper, mean_val = bootstrap_confidence_interval(
            accuracy_array, lambda x: np.mean(x), confidence=0.95
        )
        
        print(f"{name}: {mean_val:.4f} [{lower:.4f}, {upper:.4f}]")
    
    print(f"\n4. INTERPRETATION GUIDE")
    print("-" * 25)
    print("P-value < 0.05: Statistically significant difference")
    print("Cohen's d: 0.2=small, 0.5=medium, 0.8=large effect")
    print("Confidence intervals: 95% bootstrap confidence intervals")
    print("Non-overlapping CIs suggest significant difference")
    
    return df_metrics

# Example usage for single model analysis
print("\n" + "="*60)
print("STATISTICAL VALIDATION FOR CURRENT MODEL")
print("="*60)

# Perform statistical analysis on current results
stats_df = statistical_validation_analysis([results], ["Current_Model"])

# Example usage for multiple models (uncomment when you have multiple result sets)
"""
# Example for comparing multiple models:
# results_model2 = evaluate_excel_data("path_to_model2_results.xlsx")
# results_model3 = evaluate_excel_data("path_to_model3_results.xlsx")

# stats_df = statistical_validation_analysis(
#     [results, results_model2, results_model3], 
#     ["Model_A", "Model_B", "Model_C"]
# )
"""

print(f"\nAnalysis completed for {len(results)} samples.")
print("For multiple model comparison, provide additional result sets to the function.")


STATISTICAL VALIDATION FOR CURRENT MODEL
STATISTICAL VALIDATION ANALYSIS

1. PERFORMANCE SUMMARY
------------------------------
           model  accuracy  precision  recall  f1_score  tool_detection_rate  \
0  Current_Model    0.6783     0.4568  0.9487    0.6167               0.9021   

   tool_selection_robustness  parameter_completeness_rate  sample_size  
0                     0.9877                       0.4625          143  

2. BOOTSTRAP CONFIDENCE INTERVALS (95%) - Current_Model
--------------------------------------------------
Accuracy: 0.6791 [0.6014, 0.7622]
Tool Detection Rate: 0.9021
Tool Selection Robustness: 0.9877
Parameter Completeness: 0.4625

Analysis completed for 143 samples.
For multiple model comparison, provide additional result sets to the function.


# AST based evaluation for Model responses Single-turn Single/Parallel tool call

# Remove <tool_call> Tags